# Reproducible charts and live refresh

> **Krisk 0.9 notebook mode**
>
> This notebook runs locally with pandas and Krisk only. It does not require the Krisk
> server, MCP, SQLite, or PostgreSQL. Chart output embeds its renderer and can be used
> offline. The typed `ChartSpec` API is recommended for new work; legacy `krisk.plot`
> calls remain supported for existing notebooks.

There are two refresh models:

1. legacy notebook methods update a displayed chart in the current kernel;
2. living research reruns a saved query on the Krisk server while preserving an
   immutable point-in-time snapshot.

This notebook requires no server and demonstrates the first model.

In [ ]:
from pathlib import Path

import pandas as pd
import krisk.plot as kk
from krisk import Chart, ChartSpec, __version__

ROOT = Path.cwd()
if not (ROOT / "krisk" / "tests" / "data").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "krisk" / "tests" / "data"

print(f"Krisk {__version__}")

In [ ]:
gapminder = pd.read_csv(DATA_DIR / "gapminderDataFiveYear.txt", sep="\t")

## Resync a displayed chart

In [ ]:
chart = kk.bar(
    gapminder[gapminder.year == 1952],
    "continent",
    y="pop",
    how="mean",
)
chart.set_size(width=800)

In [ ]:
chart.resync_data(gapminder[gapminder.year == 2007])

`resync_data` recomputes the option and targets the existing chart element. It is
useful inside widget callbacks, but it does not create durable historical evidence.

In [ ]:
def show_year(year: int):
    """Update the existing chart from a notebook callback."""
    return chart.resync_data(gapminder[gapminder.year == year])

show_year(1997)

## Replace the chart or its data

In [ ]:
chart.replot(kk.line(gapminder, "continent"))

In [ ]:
chart.read_df(gapminder[gapminder.year == 2007])

## Create a durable static artifact

In [ ]:
snapshot = Chart.from_dataframe(
    gapminder[gapminder.year == 2007],
    ChartSpec(
        kind="scatter",
        x="lifeExp",
        y="gdpPercap",
        color="continent",
        size="pop",
        title="Gapminder in 2007",
        description="Immutable point-in-time notebook export",
    ),
)
output = ROOT / "var" / "krisk" / "gapminder-2007.html"
output.parent.mkdir(parents=True, exist_ok=True)
snapshot.to_html(output)
output

For a true **Snapshot / Live** report, use the MCP research workflow described in
`docs/tutorial.md`. The exported snapshot remains available when the server or source
is offline; the Live tab fetches current data separately.